In [ ]:
!pip install beautifulsoup4 requests tqdm -q

In [ ]:
dataset_urls = {
    'train_input': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/train/sat/index.html',
    'train_target': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/train/map/index.html',
    'valid_input': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/valid/sat/index.html',
    'valid_target': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/valid/map/index.html',
    'test_input': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/index.html',
    'test_target': 'https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/map/index.html',
}

base_save_dir = '/content/drive/MyDrive/MassachusettsRoadsDataset'

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import time
from urllib.parse import urljoin

Download Functions

In [ ]:
def get_download_links(page_url):

    try:
        print(f"  Fetching links from: {page_url}")
        response = requests.get(page_url, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        links = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if href.endswith('.tiff') or href.endswith('.tif'):
                full_url = urljoin(page_url, href)
                links.append(full_url)

        print(f"  Found {len(links)} files")
        return links
    except Exception as e:
        print(f"  Error: {e}")
        return []

def download_file(url, save_path, max_retries=3):

    for attempt in range(max_retries):
        try:
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()

            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            return True
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return False
    return False

def download_from_links(links, save_dir, dataset_name):

    print(f"\n Downloading: {dataset_name}")
    print(f"Total files: {len(links)}")
    print(f"Save to: {save_dir}")

    os.makedirs(save_dir, exist_ok=True)

    successful = 0
    failed = []
    skipped = 0

    for url in tqdm(links, desc=dataset_name):
        filename = os.path.basename(url).split('?')[0]  # Remove query parameters
        save_path = os.path.join(save_dir, filename)

        # Skip if already downloaded
        if os.path.exists(save_path):
            skipped += 1
            continue

        if download_file(url, save_path):
            successful += 1
        else:
            failed.append(filename)

        time.sleep(0.1)  # Be nice to the server

    print(f"\nResults:")
    print(f"  ✓ Downloaded: {successful}")
    print(f"  ↷ Skipped (already exist): {skipped}")
    print(f"  ✗ Failed: {len(failed)}")

    return successful, failed

Download All Datasets

In [ ]:
dirs = {
    'train_input': os.path.join(base_save_dir, 'train', 'input'),
    'train_target': os.path.join(base_save_dir, 'train', 'target'),
    'valid_input': os.path.join(base_save_dir, 'valid', 'input'),
    'valid_target': os.path.join(base_save_dir, 'valid', 'target'),
    'test_input': os.path.join(base_save_dir, 'test', 'input'),
    'test_target': os.path.join(base_save_dir, 'test', 'target'),
}

all_failed = []
total_downloaded = 0

for key, url in dataset_urls.items():
    if 'REPLACE_WITH' in url:
        print(f"\n Skipping {key} - URL not configured")
        continue

    print(f"\n Processing {key}")
    links = get_download_links(url)

    if links:
        downloaded, failed = download_from_links(links, dirs[key], key.replace('_', ' ').title())
        total_downloaded += downloaded
        all_failed.extend(failed)
    else:
        print(f"   No links found")


 Processing train_input
  Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_roads/train/sat/index.html
  Found 1108 files

 Downloading: Train Input
Total files: 1108
Save to: /content/drive/MyDrive/MassachusettsRoadsDataset/train/input


Train Input: 100%|██████████| 1108/1108 [09:28<00:00,  1.95it/s]



Results:
  ✓ Downloaded: 1108
  ↷ Skipped (already exist): 0
  ✗ Failed: 0

 Processing train_target
  Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_roads/train/map/index.html
  Found 1108 files

 Downloading: Train Target
Total files: 1108
Save to: /content/drive/MyDrive/MassachusettsRoadsDataset/train/target


Train Target: 100%|██████████| 1108/1108 [44:26<00:00,  2.41s/it]



Results:
  ✓ Downloaded: 1106
  ↷ Skipped (already exist): 0
  ✗ Failed: 2

 Processing valid_input
  Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_roads/valid/sat/index.html
  Found 14 files

 Downloading: Valid Input
Total files: 14
Save to: /content/drive/MyDrive/MassachusettsRoadsDataset/valid/input


...


Results:
  ✓ Downloaded: 14
  ↷ Skipped (already exist): 0
  ✗ Failed: 0

 Processing test_input
  Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/sat/index.html
  Found 49 files

 Downloading: Test Input
Total files: 49
Save to: /content/drive/MyDrive/MassachusettsRoadsDataset/test/input


Test Input: 100%|██████████| 49/49 [00:27<00:00,  1.78it/s]



Results:
  ✓ Downloaded: 49
  ↷ Skipped (already exist): 0
  ✗ Failed: 0

 Processing test_target
  Fetching links from: https://www.cs.toronto.edu/~vmnih/data/mass_roads/test/map/index.html
  Found 49 files

 Downloading: Test Target
Total files: 49
Save to: /content/drive/MyDrive/MassachusettsRoadsDataset/test/target


Test Target: 100%|██████████| 49/49 [00:19<00:00,  2.47it/s]


Results:
  ✓ Downloaded: 49
  ↷ Skipped (already exist): 0
  ✗ Failed: 0


In [ ]:
print(f"Total files downloaded: {total_downloaded}")
print(f"Dataset location: {base_save_dir}")
print(f"\n Failed downloads: {all_failed}")

Total files downloaded: 2340
Dataset location: /content/drive/MyDrive/MassachusettsRoadsDataset

 Failed downloads: ['25679260_15.tif', '26578735_15.tif']
